# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Name: {getattr(metadata, 'name', None)}")
print(f"Description: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

This step lists the record sets found in the metadata and displays the fields/columns for each. For all references, we use the `@id` attribute.

In [ ]:
# Helper functions to explore Croissant structure
from collections import defaultdict

def get_all_record_set_objs(dataset):
    record_sets = []
    # `dataset.metadata.recordSet` is a list of `RecordSet` objects
    if hasattr(dataset.metadata, 'recordSet'):
        recsets = getattr(dataset.metadata, 'recordSet', [])
        if not isinstance(recsets, list):
            recsets = [recsets]
        for rs in recsets:
            record_sets.append(rs)
    return record_sets

record_sets = get_all_record_set_objs(dataset)
if not record_sets:
    print('No record sets found in the metadata. Please check the schema for `recordSet` entries.')
else:
    for rs in record_sets:
        rs_id = getattr(rs, '@id', None)
        rs_name = getattr(rs, 'name', None)
        print(f'\nRecordSet @id: {rs_id}   name: {rs_name}')
        if hasattr(rs, 'field'):
            fields_list = getattr(rs, 'field')
            if not isinstance(fields_list, list):
                fields_list = [fields_list]
            for f in fields_list:
                fid = getattr(f, '@id', None)
                fname = getattr(f, 'name', None)
                ftype = getattr(f, 'dataType', None)
                print(f'  Field @id: {fid}   name: {fname}   dataType: {ftype}')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# As an example, we use the first available record set.
record_set_ids = []
for rs in get_all_record_set_objs(dataset):
    rs_id = getattr(rs, '@id', None)
    record_set_ids.append(rs_id)

if not record_set_ids:
    raise ValueError('No record sets found. Data extraction cannot proceed.')

# We'll just use the first record set for this analysis
selected_record_set_id = record_set_ids[0]

dataframes = {}
for rs_id in record_set_ids:
    # Records are a generator of dicts
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records from RecordSet {rs_id}")
        else:
            print(f"No records found for RecordSet {rs_id}")
    except Exception as e:
        print(f"Failed to load records for {rs_id}: {e}")

if selected_record_set_id in dataframes:
    df0 = dataframes[selected_record_set_id]
    print(f"\nColumns in RecordSet {selected_record_set_id}:")
    print(df0.columns.tolist())
    display(df0.head())
else:
    print(f"RecordSet {selected_record_set_id} could not be loaded into a DataFrame.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's demonstrate filtering and normalization using a numeric field (for example, `MW` for molecular weight or any other numeric field found in the selected record set). All references are by `@id` or dataset column name.

In [ ]:
# Identify a numeric field (example: 'MW' or 'coverage_percent' if exists)
numeric_field = None
numeric_fallbacks = ['MW', 'molecular_weight', 'coverage_percent', 'coverage', 'peptide_count']
for col in dataframes[selected_record_set_id].columns:
    if col in numeric_fallbacks:
        numeric_field = col
        break

if numeric_field is None:
    # Try to find any column with likely numeric data by checking dtypes
    for col in dataframes[selected_record_set_id].columns:
        try:
            # Try to convert first value to float
            test_value = dataframes[selected_record_set_id][col].iloc[0]
            float(test_value)
            numeric_field = col
            break
        except Exception:
            continue

if numeric_field is None:
    raise ValueError("No numeric field could be found for EDA.")

print(f"Using numeric field: {numeric_field}")

df = dataframes[selected_record_set_id].copy()
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
df = df.dropna(subset=[numeric_field])

threshold = df[numeric_field].mean()  # Example: mean as threshold
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} rows")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by a likely categorical field
group_field = None
cat_fallbacks = ['description', 'protein', 'accession', 'sample', 'group']
for col in df.columns:
    if col in cat_fallbacks:
        group_field = col
        break

if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"Grouped data by {group_field} (showing mean {numeric_field} per group):")
    display(grouped_df.head())
else:
    print("No suitable categorical field found to group by.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Histogram of the numeric field
plt.figure(figsize=(8, 5))
plt.hist(df[numeric_field], bins=30, edgecolor='k', alpha=0.7)
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.title(f'Distribution of {numeric_field}')
plt.show()

# If group_field exists and is categorical, plot mean
if group_field and group_field in filtered_df.columns:
    means = filtered_df.groupby(group_field)[numeric_field].mean().sort_values()
    if len(means) <= 20:
        plt.figure(figsize=(10, 6))
        means.plot(kind='bar')
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.ylabel(f'Mean {numeric_field}')
        plt.xlabel(group_field)
        plt.show()
    else:
        print(f"Too many groups to plot a bar chart for {group_field}.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² dataset was loaded and explored using the `mlcroissant` library.
- Key fields and distributions were reviewed and visualized, with filtering and normalization applied to a numeric attribute.
- The structure defined via Croissant `@id` ensures reproducible, interpretable data processing pipelines.

You may further explore the data by examining additional record sets or customizing the analysis to your research question.